# Qwen3-VL-Reranker-2B Feasibility Check for CIR

This notebook tests whether **Qwen/Qwen3-VL-Reranker-2B** is a practical smaller reranker candidate for the CIR thesis pipeline.

Immediate goal in this notebook:
1. verify runtime/hardware feasibility (especially RTX constraints),
2. run one direct CIR-style score,
3. run one tiny candidate-pool reranking test.

This is a lightweight feasibility study only (no full repo integration yet).

In [1]:
# Optional install cell (run only if needed).
# Official page recommendations:
#   transformers>=4.57.0, qwen-vl-utils>=0.0.14, torch==2.8.0
#
# Uncomment and run if your environment is missing dependencies.
# %pip install -q "transformers>=4.57.0" "qwen-vl-utils>=0.0.14" "accelerate>=0.34.0"
# %pip install -q "git+https://github.com/QwenLM/Qwen3-VL-Embedding.git"

## 2) Environment Check

In [2]:
import importlib
import torch
import transformers

def _get_module_version(name: str):
    try:
        m = importlib.import_module(name)
        return getattr(m, "__version__", "unknown")
    except Exception:
        return None

torch_version = torch.__version__
transformers_version = transformers.__version__
qwen_vl_utils_version = _get_module_version("qwen_vl_utils")

cuda_available = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if cuda_available else None
total_mem_gb = None
free_mem_gb = None
if cuda_available:
    props = torch.cuda.get_device_properties(0)
    total_mem_gb = props.total_memory / (1024**3)
    free_mem_bytes, total_mem_bytes = torch.cuda.mem_get_info(0)
    free_mem_gb = free_mem_bytes / (1024**3)

print("=== Runtime ===")
print(f"torch version: {torch_version}")
print(f"transformers version: {transformers_version}")
print(f"qwen-vl-utils version: {qwen_vl_utils_version}")
print(f"CUDA available: {cuda_available}")
print(f"GPU: {gpu_name}")
print(f"GPU total memory (GB): {None if total_mem_gb is None else round(total_mem_gb, 2)}")
print(f"GPU free memory now (GB): {None if free_mem_gb is None else round(free_mem_gb, 2)}")

print("\n=== Official recommendation check (model page) ===")
print("Recommended: transformers>=4.57.0, qwen-vl-utils>=0.0.14, torch==2.8.0")
print(f"Current torch: {torch_version} (exact 2.8.0? {'yes' if torch_version.startswith('2.8.0') else 'no'})")
print(f"Current transformers: {transformers_version} (>=4.57.0 expected)")
print(f"Current qwen-vl-utils: {qwen_vl_utils_version} (>=0.0.14 expected)")

/nfs/home/maatouk/multimodal-rag-cir/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


=== Runtime ===
torch version: 2.11.0+cu128
transformers version: 4.57.1
qwen-vl-utils version: unknown
CUDA available: False
GPU: None
GPU total memory (GB): None
GPU free memory now (GB): None

=== Official recommendation check (model page) ===
Recommended: transformers>=4.57.0, qwen-vl-utils>=0.0.14, torch==2.8.0
Current torch: 2.11.0+cu128 (exact 2.8.0? no)
Current transformers: 4.57.1 (>=4.57.0 expected)
Current qwen-vl-utils: unknown (>=0.0.14 expected)


## 3) Model Load Test

In [3]:
from __future__ import annotations

import importlib
import importlib.util
import inspect
import subprocess
import sys
import traceback
from dataclasses import dataclass
from pathlib import Path
from typing import Any

import torch

MODEL_ID = "Qwen/Qwen3-VL-Reranker-2B"


def _resolve_repo_root() -> Path:
    candidates = [Path.cwd(), Path.cwd().parent]
    for c in candidates:
        if (c / "src").exists() and (c / "notebooks").exists():
            return c
    return Path.cwd().parent


def _has_flash_attn() -> bool:
    return importlib.util.find_spec("flash_attn") is not None


def _pick_dtype() -> torch.dtype:
    if torch.cuda.is_available():
        if torch.cuda.is_bf16_supported():
            return torch.bfloat16
        return torch.float16
    return torch.float32


def _is_oom_error(exc: Exception) -> bool:
    text = str(exc).lower()
    return "out of memory" in text or "cuda out of memory" in text


def _try_import_class(module_name: str, class_name: str):
    try:
        module = importlib.import_module(module_name)
        if hasattr(module, class_name):
            return getattr(module, class_name), None
    except Exception as exc:
        return None, str(exc)
    return None, None


def _try_import_class_from_file(file_path: Path, class_name: str):
    try:
        spec = importlib.util.spec_from_file_location("_qwen3_vl_reranker_helper", str(file_path))
        if spec is None or spec.loader is None:
            return None, "Could not create module spec."
        module = importlib.util.module_from_spec(spec)
        spec.loader.exec_module(module)
        if hasattr(module, class_name):
            return getattr(module, class_name), None
        return None, f"Class {class_name} not found in file {file_path}."
    except Exception as exc:
        return None, str(exc)


def _resolve_official_class(repo_root: Path):
    class_name = "Qwen3VLReranker"
    attempts = [
        "scripts.qwen3_vl_reranker",
        "src.models.qwen3_vl_reranker",
    ]

    for module_name in attempts:
        klass, err = _try_import_class(module_name, class_name)
        if klass is not None:
            return klass, f"{module_name}.{class_name}", None

    helper_repo = repo_root / ".cache" / "Qwen3-VL-Embedding"
    clone_warning = None
    if not helper_repo.exists():
        helper_repo.parent.mkdir(parents=True, exist_ok=True)
        try:
            subprocess.run(
                ["git", "clone", "--depth", "1", "https://github.com/QwenLM/Qwen3-VL-Embedding.git", str(helper_repo)],
                check=True,
                capture_output=True,
                text=True,
            )
        except Exception as exc:
            clone_warning = f"Could not clone helper repo automatically: {exc}"

    helper_file = helper_repo / "src" / "models" / "qwen3_vl_reranker.py"
    if helper_file.exists():
        klass, err = _try_import_class_from_file(helper_file, class_name)
        if klass is not None:
            return klass, f"{helper_file}:{class_name}", clone_warning
        warn = clone_warning or "Failed to import helper class from cloned file."
        return None, None, f"{warn} Error: {err}"

    warn = clone_warning or f"Helper file not found: {helper_file}"
    return None, None, warn


@dataclass
class ModelState:
    ok: bool
    model: Any | None
    backend: str | None
    load_warning: str | None
    error: str | None


repo_root = _resolve_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

dtype = _pick_dtype()
flash_available = _has_flash_attn()
attn_impl = "flash_attention_2" if flash_available and torch.cuda.is_available() else None

print("dtype selected:", dtype)
print("flash_attention_2 available:", flash_available)
print("attn implementation requested:", attn_impl)

state = ModelState(ok=False, model=None, backend=None, load_warning=None, error=None)

RerankerClass, import_path, helper_warning = _resolve_official_class(repo_root)

if RerankerClass is None:
    state.error = "Official Qwen3VLReranker helper could not be resolved."
    state.load_warning = helper_warning
else:
    try:
        init_sig = inspect.signature(RerankerClass.__init__)
        init_params = set(init_sig.parameters.keys())

        kwargs: dict[str, Any] = {"model_name_or_path": MODEL_ID}
        if "torch_dtype" in init_params:
            kwargs["torch_dtype"] = dtype
        elif "dtype" in init_params:
            kwargs["dtype"] = dtype

        if attn_impl is not None and "attn_implementation" in init_params:
            kwargs["attn_implementation"] = attn_impl

        model = RerankerClass(**kwargs)
        state.ok = True
        state.model = model
        state.backend = f"official_helper:{import_path}"
        state.load_warning = helper_warning
    except Exception as exc:
        if _is_oom_error(exc):
            state.error = f"OOM during official helper load: {exc}"
        else:
            state.error = "Official helper load failed:\n" + traceback.format_exc()

print("\n=== Load result ===")
print("model loading succeeded:", state.ok)
print("backend:", state.backend)
if state.model is not None:
    print("model class:", type(state.model).__name__)
    if hasattr(state.model, "device"):
        print("device:", state.model.device)
    elif hasattr(state.model, "model"):
        try:
            print("device:", next(state.model.model.parameters()).device)
        except Exception:
            print("device: unknown")

if state.load_warning:
    print("warning:", state.load_warning)
if state.error:
    print("error:", state.error)

dtype selected: torch.float32
flash_attention_2 available: False
attn implementation requested: None

=== Load result ===
model loading succeeded: True
backend: official_helper:/nfs/home/maatouk/multimodal-rag-cir/.cache/Qwen3-VL-Embedding/src/models/qwen3_vl_reranker.py:Qwen3VLReranker
model class: Qwen3VLReranker
device: cpu


## 4) One Direct CIR-Style Score

In [4]:
from pathlib import Path
import os
import sys
import traceback

# Ensure project root import and working directory are correct.
project_candidates = [Path.cwd(), Path.cwd().parent]
for candidate in project_candidates:
    if (candidate / "src").exists() and (candidate / "data").exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

if "repo_root" in globals():
    os.chdir(str(repo_root))
else:
    for candidate in project_candidates:
        if (candidate / "src").exists() and (candidate / "data").exists():
            os.chdir(str(candidate))
            break

from src.datasets.cirr import build_cirr_dataset

single_score_ok = False
single_score_value = None
single_score_error = None

if not state.ok:
    print("Model is not loaded, skipping direct score.")
else:
    try:
        print("cwd:", os.getcwd())
        triplet_ds = build_cirr_dataset(split="val", mode="triplets", image_transform=None, caption_transform=None)
        image_ds = build_cirr_dataset(split="val", mode="images", image_transform=None, caption_transform=None)

        sample = triplet_ds[0]
        reference_name = sample["reference_name"]
        text_edit = sample["caption"]
        target_name = sample["target_name"]

        reference_rel = image_ds.name_to_relpath[reference_name]
        reference_path = str(Path(image_ds.images_dirpath) / reference_rel)

        target_rel = image_ds.name_to_relpath[target_name]
        target_path = str(Path(image_ds.images_dirpath) / target_rel)

        instruction = (
            "Given a reference image and a text modification in the query, "
            "score candidate images by how well they match the edited target intent."
        )

        inputs = {
            "instruction": instruction,
            "query": {
                "text": f"Reference image with modification: {text_edit}",
                "image": reference_path,
            },
            "documents": [
                {"image": target_path}
            ],
            "fps": 1.0,
        }

        scores = state.model.process(inputs)
        single_score_value = float(scores[0])
        single_score_ok = True

        print("direct score succeeded: True")
        print("reference sample pair_id:", sample["pair_id"])
        print("target candidate:", target_name)
        print("returned score:", single_score_value)
    except Exception as exc:
        if _is_oom_error(exc):
            single_score_error = f"OOM during direct score: {exc}"
        else:
            single_score_error = "Direct score failed:\n" + traceback.format_exc()

        print("direct score succeeded: False")
        print(single_score_error)

You're using a Qwen2TokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


cwd: /nfs/home/maatouk/multimodal-rag-cir


/nfs/home/maatouk/multimodal-rag-cir/.venv/lib/python3.12/site-packages/transformers/tokenization_utils_base.py:2779: UserWarning: `max_length` is ignored when `padding`=`True` and there is no truncation strategy. To pad to max length, use `padding='max_length'`.
  warnings.warn(


direct score succeeded: True
reference sample pair_id: 12060
target candidate: dev-1028-1-img1
returned score: 0.607456386089325


## 5) Tiny Candidate-Pool Test

In [5]:
import os
import random
import traceback

tiny_pool_ok = False
tiny_pool_error = None
tiny_pool_rows = []

if not state.ok:
    print("Model is not loaded, skipping tiny candidate-pool test.")
else:
    try:
        if "repo_root" in globals():
            os.chdir(str(repo_root))

        if "triplet_ds" not in globals() or "image_ds" not in globals():
            from src.datasets.cirr import build_cirr_dataset
            triplet_ds = build_cirr_dataset(split="val", mode="triplets", image_transform=None, caption_transform=None)
            image_ds = build_cirr_dataset(split="val", mode="images", image_transform=None, caption_transform=None)

        rng = random.Random(42)
        sample = triplet_ds[0]

        reference_name = sample["reference_name"]
        target_name = sample["target_name"]
        text_edit = sample["caption"]

        reference_rel = image_ds.name_to_relpath[reference_name]
        reference_path = str(Path(image_ds.images_dirpath) / reference_rel)

        all_ids = list(image_ds.name_to_relpath.keys())
        distractor_pool = [cid for cid in all_ids if cid not in {reference_name, target_name}]
        distractor_ids = rng.sample(distractor_pool, k=min(3, len(distractor_pool)))

        candidate_ids = [target_name] + distractor_ids
        rng.shuffle(candidate_ids)

        documents = []
        for cid in candidate_ids:
            rel = image_ds.name_to_relpath[cid]
            path = str(Path(image_ds.images_dirpath) / rel)
            documents.append({"image": path})

        instruction = (
            "Given a reference image and a text modification in the query, "
            "score each candidate image by relevance to the edited target."
        )

        inputs = {
            "instruction": instruction,
            "query": {
                "text": f"Reference image with modification: {text_edit}",
                "image": reference_path,
            },
            "documents": documents,
            "fps": 1.0,
        }

        scores = [float(v) for v in state.model.process(inputs)]

        for cid, score in zip(candidate_ids, scores):
            tiny_pool_rows.append({
                "candidate_id": cid,
                "score": score,
                "is_target": (cid == target_name),
            })

        tiny_pool_rows = sorted(tiny_pool_rows, key=lambda x: x["score"], reverse=True)
        tiny_pool_ok = True

        print("tiny pool test succeeded: True")
        print("Raw scores (input order):")
        for cid, score in zip(candidate_ids, scores):
            marker = "<TARGET>" if cid == target_name else ""
            print(f"  {cid}: {score:.6f} {marker}")

        print("\nSorted ranking (desc):")
        for rank, row in enumerate(tiny_pool_rows, start=1):
            marker = "<TARGET>" if row["is_target"] else ""
            print(f"  {rank:02d}. {row['candidate_id']} | score={row['score']:.6f} {marker}")
    except Exception as exc:
        if _is_oom_error(exc):
            tiny_pool_error = f"OOM during tiny candidate-pool test: {exc}"
        else:
            tiny_pool_error = "Tiny candidate-pool test failed:\n" + traceback.format_exc()

        print("tiny pool test succeeded: False")
        print(tiny_pool_error)

tiny pool test succeeded: True
Raw scores (input order):
  dev-238-2-img0: 0.048890 
  dev-835-2-img1: 0.103884 
  dev-1028-1-img1: 0.594284 <TARGET>
  dev-184-2-img0: 0.028322 

Sorted ranking (desc):
  01. dev-1028-1-img1 | score=0.594284 <TARGET>
  02. dev-835-2-img1 | score=0.103884 
  03. dev-238-2-img0 | score=0.048890 
  04. dev-184-2-img0 | score=0.028322 


## 6) Final Notebook Summary

In [6]:
summary = {
    "model_loaded": bool(state.ok),
    "backend": state.backend,
    "single_score_worked": bool(single_score_ok),
    "tiny_pool_worked": bool(tiny_pool_ok),
    "single_score_value": single_score_value,
}

print("=== Feasibility Conclusion ===")
print(f"Model loaded: {summary['model_loaded']}")
print(f"Backend path: {summary['backend']}")
print(f"Direct CIR-style score worked: {summary['single_score_worked']}")
print(f"Tiny candidate-pool ranking worked: {summary['tiny_pool_worked']}")
print(f"Direct score value: {summary['single_score_value']}")

if state.error:
    print("\nLoad error detail:")
    print(state.error)
if single_score_error:
    print("\nDirect score error detail:")
    print(single_score_error)
if tiny_pool_error:
    print("\nTiny pool error detail:")
    print(tiny_pool_error)

if summary['model_loaded'] and summary['single_score_worked']:
    print("\nPracticality signal: positive for feasibility; candidate is reasonable to consider for next reranker experiments.")
else:
    print("\nPracticality signal: blocked by environment/load/runtime issue; inspect error details before integration decisions.")

=== Feasibility Conclusion ===
Model loaded: True
Backend path: official_helper:/nfs/home/maatouk/multimodal-rag-cir/.cache/Qwen3-VL-Embedding/src/models/qwen3_vl_reranker.py:Qwen3VLReranker
Direct CIR-style score worked: True
Tiny candidate-pool ranking worked: True
Direct score value: 0.607456386089325

Practicality signal: positive for feasibility; candidate is reasonable to consider for next reranker experiments.
